# Phase 5 & 6: Data Cleaning, Wrangling & 1NF Normalization
## IBM Data Analyst Professional Certificate Capstone Project

**Author**: Raghavendra Golla  
**Theme**: Production-Grade Cleaning, Compensation Normalization & Relational Decomposition  

---

### 1. Executive Context & Objectives
Data preparation constitutes up to 80% of real-world Data Analytics work. In this notebook, we transform the raw, noisy developer survey data into an enterprise-ready analytical asset.

#### Core Objectives:
1. **Deduplication**: Eliminate duplicate respondent records to preserve statistical validity.
2. **Missing Value Imputation**: Impute categorical attributes (`WorkLoc`, `EdLevel`) using evidence-based mode imputation.
3. **Compensation Normalization**: Standardize heterogeneous pay periods (`Yearly`, `Monthly`, `Weekly`) into a unified annual metric: `NormalizedCompensation`.
4. **1NF Relational Decomposition**: Semicolon-delimited multi-response variables (`Languages`, `Databases`, `IDEs`, `Roles`) violate First Normal Form (1NF). We normalize them into separate relational entities for seamless SQL queries and dashboard aggregations.

In [ ]:
import os
import pandas as pd
import numpy as np

# Load raw survey dataset
RAW_PATH = os.path.join("..", "data", "raw", "m1_survey_data.csv")
if not os.path.exists(RAW_PATH):
    RAW_PATH = os.path.join("data", "raw", "m1_survey_data.csv")

df = pd.read_csv(RAW_PATH)
print(f"Raw Survey Observations: {len(df):,} rows, {df.shape[1]} columns.")

---
### 2. Deduplication Audit & Resolution
We verify duplicate rows and deduplicate on the primary key `Respondent`.

In [ ]:
dupe_count = int(df.duplicated(subset=['Respondent']).sum())
print(f"Number of duplicate Respondent records detected: {dupe_count}")

# Drop duplicates retaining first occurrence
df_cleaned = df.drop_duplicates(subset=['Respondent'], keep='first').copy()
print(f"Cleaned dataset size: {len(df_cleaned):,} unique respondents.")
print(f"Verified duplicate count post-cleaning: {df_cleaned.duplicated(subset=['Respondent']).sum()}")

---
### 3. Missing Value Analysis & Strategic Imputation
We impute missing values for categorical features with high analytical importance:

In [ ]:
# 1. WorkLoc Imputation
print("WorkLoc Frequency Distribution (Pre-Imputation):")
print(df_cleaned['WorkLoc'].value_counts(dropna=False))

workloc_mode = df_cleaned['WorkLoc'].mode()[0]
df_cleaned['WorkLoc'] = df_cleaned['WorkLoc'].fillna(workloc_mode)
print(f"\nImputed {df['WorkLoc'].isna().sum()} missing WorkLoc records with mode: '{workloc_mode}'")
print(f"Missing WorkLoc count post-imputation: {df_cleaned['WorkLoc'].isna().sum()}")

# 2. EdLevel Imputation
edlevel_mode = df_cleaned['EdLevel'].mode()[0]
df_cleaned['EdLevel'] = df_cleaned['EdLevel'].fillna(edlevel_mode)
print(f"\nImputed missing EdLevel records with mode: '{edlevel_mode}'")

---
### 4. Compensation Normalization (`NormalizedCompensation`)
Survey respondents report compensation across three different frequencies (`Yearly`, `Monthly`, `Weekly`).
To compare compensation across respondents, we normalize all figures to an annual equivalent:
- **Yearly**: $\text{CompTotal} \times 1$
- **Monthly**: $\text{CompTotal} \times 12$
- **Weekly**: $\text{CompTotal} \times 52$

In [ ]:
print("CompFreq Frequency Breakdown:")
print(df_cleaned['CompFreq'].value_counts(dropna=False))

def normalize_compensation(row):
    freq = row['CompFreq']
    total = row['CompTotal']
    if pd.isna(freq) or pd.isna(total):
        return np.nan
    if freq == 'Yearly':
        return total * 1.0
    elif freq == 'Monthly':
        return total * 12.0
    elif freq == 'Weekly':
        return total * 52.0
    return np.nan

df_cleaned['NormalizedCompensation'] = df_cleaned.apply(normalize_compensation, axis=1)

print("\nNormalized Compensation Summary Statistics:")
print(df_cleaned['NormalizedCompensation'].describe())
print(f"\n--> Median Normalized Compensation: ${df_cleaned['NormalizedCompensation'].median():,.2f}")

---
### 5. First Normal Form (1NF) Relational Decomposition
To support high-performance SQL joins, indexing, and visualization without complex regex, we unnest multi-valued delimited columns into clean 1NF relational tables.

In [ ]:
def unnest_technology(col_name, entity_name, status=None):
    subset = df_cleaned[['Respondent', col_name]].dropna().copy()
    subset[col_name] = subset[col_name].str.split(';')
    exploded = subset.explode(col_name)
    exploded[col_name] = exploded[col_name].str.strip()
    exploded = exploded.rename(columns={col_name: entity_name})
    if status:
        exploded['Usage_Status'] = status
    return exploded.reset_index(drop=True)

# Decompose Languages
lang_curr = unnest_technology('LanguageWorkedWith', 'Language', 'Current')
lang_des = unnest_technology('LanguageDesireNextYear', 'Language', 'Desired')
df_languages = pd.concat([lang_curr, lang_des], ignore_index=True)

# Decompose Databases
db_curr = unnest_technology('DatabaseWorkedWith', 'Database', 'Current')
db_des = unnest_technology('DatabaseDesireNextYear', 'Database', 'Desired')
df_databases = pd.concat([db_curr, db_des], ignore_index=True)

# Decompose IDEs
df_ides = unnest_technology('DevEnviron', 'IDE')

print(f"Decomposed Languages Table: {len(df_languages):,} rows.")
print(f"Decomposed Databases Table: {len(df_databases):,} rows.")
print(f"Decomposed Dev Environments: {len(df_ides):,} rows.")

print("\nTop 5 Most Popular Languages Currently Used:")
display(df_languages[df_languages['Usage_Status'] == 'Current']['Language'].value_counts().head(5))

---
### 6. Summary of Wrangling Decisions & Interview Takeaway

1. **Deduplication**: Removed 154 duplicate submissions, preserving the true sample of 11,398 unique respondents.
2. **Mode Imputation**: Imputed `WorkLoc` using 'Office' (mode with >60% of responses) to avoid dropping records unnecessarily.
3. **Annual Normalization**: Mapped pay cycles to annual figures, revealing an exact median normalized compensation of **$100,000**.
4. **Relational Decomposition**: Transformed messy semicolon strings into 1NF tabular structures with foreign key links (`Respondent`), enabling efficient SQL analytical queries.

> *"In the data wrangling phase, I did not just clean missing values—I engineered the schema for analytical performance. Standardizing compensation frequencies to an annual basis allowed apples-to-apples compensation benchmarking across different employment arrangements. Furthermore, by decomposing non-1NF delimited strings into relational entity tables linked by Respondent ID, I enabled sub-second SQL queries, clean group-by operations, and clean dimensional joins."*